# azure_ai_agentserver_invocations smoke test

Invocations protocol for Azure AI Hosted Agents. Provides `InvocationAgentServerHost` plus a set of handler decorators (`invoke_handler`, `get_invocation_handler`, `cancel_invocation_handler`, `ws_handler`) layered on top of `azure-ai-agentserver-core`.

In [ ]:
# Example 1: Import the primary API surface and inspect its type/MRO.
from azure.ai.agentserver.invocations import InvocationAgentServerHost

print(f"type: {type(InvocationAgentServerHost).__name__}")
print(f"qualname: {InvocationAgentServerHost.__qualname__}")
mro_names = [c.__name__ for c in InvocationAgentServerHost.__mro__]
print(f"MRO: {mro_names}")
# The host must descend from AgentServerHost in the sibling -core package.
assert any('AgentServerHost' in n for n in mro_names), "expected AgentServerHost in MRO"
print("ok")

In [ ]:
# Example 2: Instantiate the host and verify the documented decorator methods exist.
from azure.ai.agentserver.invocations import InvocationAgentServerHost

app = InvocationAgentServerHost()

expected_decorators = (
    'invoke_handler',
    'get_invocation_handler',
    'cancel_invocation_handler',
    'ws_handler',
)
for name in expected_decorators:
    attr = getattr(app, name, None)
    print(f"{name}: {attr!r}")
    assert callable(attr), f"{name} should be callable"
print("all four documented decorators are present and callable")

In [ ]:
# Example 3: Register an invoke handler via the @app.invoke_handler decorator,
# without actually starting the HTTP server. We only verify registration succeeds.
from azure.ai.agentserver.invocations import InvocationAgentServerHost

app = InvocationAgentServerHost()

@app.invoke_handler
async def handle(request):
    # Body never actually runs in this notebook - we just register the handler.
    return {"echo": await request.json()}

# The decorator returns the wrapped function (or the function itself).
assert callable(handle), "decorator should return a callable"
assert handle.__name__ == 'handle'
print(f"registered handler: {handle.__name__}")

In [ ]:
# Example 4: Inspect the public module surface and ensure the documented primary class
# is exported from the top-level package namespace.
import azure.ai.agentserver.invocations as invocations_pkg

public = sorted(n for n in dir(invocations_pkg) if not n.startswith('_'))
print(f"{len(public)} public name(s) on azure.ai.agentserver.invocations")
print("sample:", public[:20])

assert 'InvocationAgentServerHost' in public, \
    "InvocationAgentServerHost must be exposed at azure.ai.agentserver.invocations"

# Sanity-check the dependency edge: azure.ai.agentserver.core must be importable too,
# since the invocations package builds on top of it.
import azure.ai.agentserver.core
print("sibling -core package importable; runtime dep edge holds")

In [ ]:
# Example 5: Round-trip - construct a fresh host, register one of each documented handler,
# and assert the host instance remains in a usable state with the registrations attached.
from azure.ai.agentserver.invocations import InvocationAgentServerHost

app = InvocationAgentServerHost()

@app.invoke_handler
async def invoke(request):
    return {"status": "ok"}

@app.get_invocation_handler
async def get_invocation(request):
    return {"status": "running"}

@app.cancel_invocation_handler
async def cancel(request):
    return {"status": "cancelled"}

for fn in (invoke, get_invocation, cancel):
    assert callable(fn)
    assert fn.__name__
print("three handlers registered on one host instance")
assert app is not None
print("PASS")